# Ensemble Stacking Model — LSTM + GRU + TFT

**Stacking ensemble with frozen pre-trained base models and a trainable MLP meta-learner**

| Base Model | Window | Config | Checkpoint |
|:-----------|:------:|:------:|:-----------|
| LSTM       |   30   | small (hidden=32) | `w30_small.pt` |
| GRU        |   30   | small (hidden=32) | `w30_small.pt` |
| TFT        |   15   | hidden=32, attn=4 | `best-epoch003-valloss0.007141.ckpt` |

- Each base model outputs **7 quantiles** → **21 stacked features** → MLP → 1 point prediction
- All base model weights are **frozen**; only the MLP meta-learner is trained
- Meta-learner trained on **val** predictions (2025-01–06), evaluated on **test** (≥ 2025-07)
- Target: `target_pct_change` (next-day return fraction)

In [6]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
%%capture
!pip install scikit-learn matplotlib seaborn tqdm pytorch-forecasting lightning

In [8]:
import os, sys, json, math, random, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score
)

import lightning.pytorch as pl
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')

Device: cuda
GPU   : Tesla T4
VRAM  : 15.6 GB
PyTorch: 2.10.0+cu128


In [9]:
# ── Paths to pre-trained model weights ────────────────────────────────
GRU_CKPT  = Path('/content/drive/MyDrive/inlp/GRU_outputs/gru_outputs/checkpoints/w30_small.pt')
LSTM_CKPT = Path('/content/drive/MyDrive/inlp/lstm_outputs/lstm_outputs-2/checkpoints/w30_small.pt')
TFT_CKPT  = Path('/content/drive/MyDrive/inlp/tft/window_15/checkpoints/best-epoch003-valloss0.007141.ckpt')

# Kaggle fallbacks
if not GRU_CKPT.exists():
    GRU_CKPT = Path('/kaggle/input/model-weights/gru_outputs/checkpoints/w30_small.pt')
if not LSTM_CKPT.exists():
    LSTM_CKPT = Path('/kaggle/input/model-weights/lstm_outputs-2/checkpoints/w30_small.pt')
if not TFT_CKPT.exists():
    TFT_CKPT = Path('/kaggle/input/model-weights/tft/window_15/checkpoints/best-epoch003-valloss0.007141.ckpt')

# ── Dataset ─────────────────────────────────────────────────────
DATA_PATH = Path('/content/drive/MyDrive/inlp/dataset.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('/kaggle/input/dataset/dataset.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('dataset.csv not found!')

# ── Windows ─────────────────────────────────────────────────────
LSTM_WINDOW = 30
GRU_WINDOW  = 30
TFT_WINDOW  = 15

# ── Date splits ─────────────────────────────────────────────────
TRAIN_END  = pd.Timestamp('2024-12-31')
VAL_START  = pd.Timestamp('2025-01-01')
VAL_END    = pd.Timestamp('2025-06-30')
TEST_START = pd.Timestamp('2025-07-01')

# ── Quantiles ───────────────────────────────────────────────────
QUANTILES  = [0.10, 0.25, 0.40, 0.50, 0.60, 0.75, 0.90]
MEDIAN_IDX = 3
NUM_QUANTILES = 7

# ── Column names ─────────────────────────────────────────────────
TARGET_COL   = 'target_pct_change'
PRICE_COL    = 'Adj Close'
DATE_COL     = 'Date'
SYMBOL_COL   = 'Symbol'
TIME_IDX_COL = 'time_idx'

EXCLUDE_COLS = {
    'Date', 'Symbol', 'Open', 'High', 'Low', 'Close',
    'Adj Close', 'Volume', 'symbol_base', 'time_idx',
}

EPS = 1e-8
NUM_WORKERS = 0   # safe for Windows

# ── Meta-learner config ─────────────────────────────────────────
META_EPOCHS   = 100
META_LR       = 1e-3
META_BS       = 512
META_PATIENCE = 15

# ── Output ──────────────────────────────────────────────────────
OUTPUT_DIR = Path('ensemble_outputs')
for _sub in ['checkpoints', 'plots', 'results']:
    (OUTPUT_DIR / _sub).mkdir(parents=True, exist_ok=True)

print('Configuration loaded.')
for _p, _n in [(GRU_CKPT, 'GRU'), (LSTM_CKPT, 'LSTM'), (TFT_CKPT, 'TFT')]:
    print(f'  {_n}: {_p} | exists={_p.exists()}')
print(f'  Data: {DATA_PATH}')

Configuration loaded.
  GRU: /content/drive/MyDrive/inlp/GRU_outputs/gru_outputs/checkpoints/w30_small.pt | exists=True
  LSTM: /content/drive/MyDrive/inlp/lstm_outputs/lstm_outputs-2/checkpoints/w30_small.pt | exists=True
  TFT: /content/drive/MyDrive/inlp/tft/window_15/checkpoints/best-epoch003-valloss0.007141.ckpt | exists=True
  Data: /content/drive/MyDrive/inlp/dataset.csv


## 1. Data Loading & Feature Engineering

In [10]:
def load_and_prepare(path: Path):
    df = pd.read_csv(path)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')
    df = df.dropna(subset=[DATE_COL])
    df = df.sort_values([SYMBOL_COL, DATE_COL]).reset_index(drop=True)
    df = df.drop_duplicates([SYMBOL_COL, DATE_COL])

    df['dow']            = df[DATE_COL].dt.weekday.astype(np.float32)
    df['dom']            = df[DATE_COL].dt.day.astype(np.float32)
    df['month']          = df[DATE_COL].dt.month.astype(np.float32)
    df['is_month_start'] = df[DATE_COL].dt.is_month_start.astype(np.float32)
    df['is_month_end']   = df[DATE_COL].dt.is_month_end.astype(np.float32)

    num_cols  = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    feat_cols = [c for c in num_cols if c not in EXCLUDE_COLS]
    if TARGET_COL in feat_cols:
        feat_cols = [TARGET_COL] + [c for c in feat_cols if c != TARGET_COL]
    df[feat_cols] = df[feat_cols].fillna(0.0)

    print(f'Loaded: {df.shape[0]:,} rows | {df[SYMBOL_COL].nunique()} symbols')
    print(f'Dates : {df[DATE_COL].min().date()} to {df[DATE_COL].max().date()}')
    print(f'Features ({len(feat_cols)}): {feat_cols[:5]} ...')
    return df, feat_cols

df_raw, FEATURE_COLS = load_and_prepare(DATA_PATH)
N_FEATURES = len(FEATURE_COLS)
SYMBOLS = sorted(df_raw[SYMBOL_COL].unique())
print(f'Symbols: {len(SYMBOLS)}')

Loaded: 71,938 rows | 50 symbols
Dates : 2020-03-12 to 2026-03-27
Features (34): ['target_pct_change', 'adj_ret_1d', 'price_range_pct', 'gap_pct', 'rolling_volatility_5d'] ...
Symbols: 50


## 2. Base Model Architectures (GRU & LSTM)

In [11]:
class GRUQuantile(nn.Module):
    """3-layer stacked GRU with residual connections and quantile head."""
    def __init__(self, input_size, hidden_size=64, num_layers=2,
                 dropout=0.2, num_quantiles=7):
        super().__init__()
        self.hidden_size   = hidden_size
        self.num_layers    = num_layers
        self.num_quantiles = num_quantiles

        self.gru1  = nn.GRU(input_size,  hidden_size, num_layers=1, batch_first=True)
        self.act1  = nn.Tanh()
        self.drop1 = nn.Dropout(dropout)

        self.gru2  = nn.GRU(hidden_size, hidden_size, num_layers=1, batch_first=True)
        self.act2  = nn.Tanh()
        self.drop2 = nn.Dropout(dropout)

        self.gru3  = nn.GRU(hidden_size, hidden_size, num_layers=1, batch_first=True)
        self.act3  = nn.Tanh()
        self.drop3 = nn.Dropout(dropout)

        self.norm  = nn.LayerNorm(hidden_size)
        self.fc1   = nn.Linear(hidden_size, hidden_size // 2)
        self.gelu  = nn.GELU()
        self.drop4 = nn.Dropout(dropout)
        self.head  = nn.Linear(hidden_size // 2, num_quantiles)

    def forward(self, x):
        out, _ = self.gru1(x)
        out    = self.drop1(self.act1(out))
        res    = out
        out, _ = self.gru2(out)
        out    = self.drop2(self.act2(out)) + res
        res    = out
        out, _ = self.gru3(out)
        out    = self.drop3(self.act3(out)) + res
        last   = self.norm(out[:, -1, :])
        h      = self.drop4(self.gelu(self.fc1(last)))
        return  self.head(h)


class LSTMQuantile(nn.Module):
    """3-layer stacked LSTM with residual connections and quantile head."""
    def __init__(self, input_size, hidden_size=64, num_layers=2,
                 dropout=0.2, num_quantiles=7):
        super().__init__()
        self.hidden_size   = hidden_size
        self.num_layers    = num_layers
        self.num_quantiles = num_quantiles

        self.lstm1 = nn.LSTM(input_size,  hidden_size, num_layers=1, batch_first=True)
        self.act1  = nn.Tanh()
        self.drop1 = nn.Dropout(dropout)

        self.lstm2 = nn.LSTM(hidden_size, hidden_size, num_layers=1, batch_first=True)
        self.act2  = nn.Tanh()
        self.drop2 = nn.Dropout(dropout)

        self.lstm3 = nn.LSTM(hidden_size, hidden_size, num_layers=1, batch_first=True)
        self.act3  = nn.Tanh()
        self.drop3 = nn.Dropout(dropout)

        self.norm  = nn.LayerNorm(hidden_size)
        self.fc1   = nn.Linear(hidden_size, hidden_size // 2)
        self.gelu  = nn.GELU()
        self.drop4 = nn.Dropout(dropout)
        self.head  = nn.Linear(hidden_size // 2, num_quantiles)

    def forward(self, x):
        out, _ = self.lstm1(x)
        out    = self.drop1(self.act1(out))
        res    = out
        out, _ = self.lstm2(out)
        out    = self.drop2(self.act2(out)) + res
        res    = out
        out, _ = self.lstm3(out)
        out    = self.drop3(self.act3(out)) + res
        last   = self.norm(out[:, -1, :])
        h      = self.drop4(self.gelu(self.fc1(last)))
        return  self.head(h)

print('GRU & LSTM architectures defined.')

GRU & LSTM architectures defined.


## 3. Sliding-Window Dataset (LSTM / GRU)

In [12]:
class StockWindowDataset(Dataset):
    """Sliding-window dataset identical to the one used for training LSTM/GRU."""
    def __init__(self, df, feature_cols, window_size, split,
                 scaler=None, fit_scaler=False):
        assert split in ('train', 'val', 'test')
        self.window_size  = window_size
        self.feature_cols = feature_cols
        xs, ys, bpxs, tpxs = [], [], [], []
        syms_list, date_list = [], []
        train_feats_list = []

        for sym, grp in df.groupby(SYMBOL_COL, sort=True):
            grp = grp.sort_values(DATE_COL).reset_index(drop=True)
            if len(grp) < window_size + 1:
                continue
            feats   = grp[feature_cols].values.astype(np.float32)
            targets = grp[TARGET_COL].values.astype(np.float32)
            prices  = grp[PRICE_COL].values.astype(np.float32)
            dt_arr  = grp[DATE_COL].values
            feats   = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)

            if fit_scaler:
                tmask = pd.DatetimeIndex(dt_arr) <= TRAIN_END
                if tmask.any():
                    train_feats_list.append(feats[tmask])

            for t in range(window_size, len(grp)):
                d = pd.Timestamp(dt_arr[t])
                in_split = {
                    'train': d <= TRAIN_END,
                    'val':   VAL_START <= d <= VAL_END,
                    'test':  d >= TEST_START,
                }[split]
                if not in_split:
                    continue
                xs.append(feats[t - window_size : t])
                ys.append(targets[t])
                bpxs.append(prices[t - 1])
                tpxs.append(prices[t])
                syms_list.append(sym)
                date_list.append(d)

        if fit_scaler and train_feats_list:
            scaler = StandardScaler()
            scaler.fit(np.vstack(train_feats_list))
        self.scaler = scaler

        if len(xs) > 0:
            xs_arr = np.stack(xs).astype(np.float32)
            if self.scaler is not None:
                N, W, F = xs_arr.shape
                xs_arr = self.scaler.transform(
                    xs_arr.reshape(-1, F)).reshape(N, W, F)
            self.xs = xs_arr
        else:
            self.xs = np.zeros((0, window_size, len(feature_cols)), np.float32)

        self.ys       = np.array(ys,       np.float32)
        self.base_pxs = np.array(bpxs,     np.float32)
        self.true_pxs = np.array(tpxs,     np.float32)
        self.symbols  = np.array(syms_list, object)
        self.dates    = np.array(date_list, object)
        n_sym = len(np.unique(self.symbols)) if len(self.symbols) > 0 else 0
        print(f'  [{split:5s}] {len(self.xs):7,} samples | '
              f'{n_sym:3d} symbols | window={window_size}')

    def __len__(self):  return len(self.xs)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.xs[idx]),
            torch.tensor(self.ys[idx],       dtype=torch.float32),
            torch.tensor(self.base_pxs[idx], dtype=torch.float32),
            torch.tensor(self.true_pxs[idx], dtype=torch.float32),
        )


def build_datasets(df, feature_cols, window_size):
    print(f'Building datasets (window={window_size}) ...')
    tr = StockWindowDataset(df, feature_cols, window_size, 'train',
                            fit_scaler=True)
    va = StockWindowDataset(df, feature_cols, window_size, 'val',
                            scaler=tr.scaler)
    te = StockWindowDataset(df, feature_cols, window_size, 'test',
                            scaler=tr.scaler)
    return tr, va, te, tr.scaler

print('Dataset class defined.')

Dataset class defined.


In [13]:
# Build window=30 datasets for LSTM and GRU
train30, val30, test30, scaler30 = build_datasets(df_raw, FEATURE_COLS, 30)

Building datasets (window=30) ...
  [train]  55,316 samples |  49 symbols | window=30
  [val  ]   6,027 samples |  49 symbols | window=30
  [test ]   9,095 samples |  50 symbols | window=30


## 4. Metrics & Inference Helpers

In [14]:
def compute_metrics(y_true, y_pred, base_px, true_px):
    """Compute regression + direction metrics (same as base model pipelines)."""
    valid = (np.isfinite(y_true) & np.isfinite(y_pred)
             & np.isfinite(base_px) & np.isfinite(true_px))
    yt, yp = y_true[valid], y_pred[valid]
    bp, tp = base_px[valid], true_px[valid]
    if len(yt) == 0:
        return {'n_samples': 0}

    pred_px  = bp * (1.0 + yp)
    denom_px = np.where(np.abs(tp) < EPS, EPS, np.abs(tp))
    mape_px  = float(np.mean(np.abs(pred_px - tp) / denom_px) * 100.0)

    err_pct = (yp - yt) * 100.0
    mse_p   = float(np.mean(err_pct ** 2))
    rmse_p  = float(np.sqrt(mse_p))
    mae_p   = float(np.mean(np.abs(err_pct)))

    true_up = (yt > 0).astype(int)
    pred_up = (yp > 0).astype(int)
    dir_acc = float(np.mean(true_up == pred_up))

    return {
        'MAPE_price':      mape_px,
        'MSE_pct':         mse_p,
        'RMSE_pct':        rmse_p,
        'MAE_pct':         mae_p,
        'F1_macro':        float(f1_score(true_up, pred_up, average='macro', zero_division=0)),
        'Accuracy':        float(accuracy_score(true_up, pred_up)),
        'Precision_macro': float(precision_score(true_up, pred_up, average='macro', zero_division=0)),
        'Recall_macro':    float(recall_score(true_up, pred_up, average='macro', zero_division=0)),
        'Directional_Acc': dir_acc,
        'n_samples':       len(yt),
        'pct_up_true':     float(np.mean(true_up)),
        'pct_up_pred':     float(np.mean(pred_up)),
    }

REPORT_COLS = [
    'MAPE_price', 'MSE_pct', 'RMSE_pct', 'MAE_pct',
    'F1_macro', 'Accuracy', 'Precision_macro', 'Recall_macro',
    'Directional_Acc', 'n_samples',
]


@torch.no_grad()
def get_all_quantile_preds(model, dataset, batch_size=512):
    """Run frozen model on dataset, return ALL 7 quantile outputs + metadata."""
    loader = DataLoader(dataset, batch_size=batch_size,
                        shuffle=False, num_workers=NUM_WORKERS)
    model.eval()
    all_preds, all_trues, all_bp, all_tp = [], [], [], []
    for x, y, bp, tp in loader:
        out = model(x.to(DEVICE))          # (B, 7)
        all_preds.append(out.cpu().numpy())
        all_trues.append(y.numpy())
        all_bp.append(bp.numpy())
        all_tp.append(tp.numpy())
    return {
        'preds_q':  np.concatenate(all_preds),
        'trues':    np.concatenate(all_trues),
        'base_px':  np.concatenate(all_bp),
        'true_px':  np.concatenate(all_tp),
        'symbols':  dataset.symbols,
        'dates':    dataset.dates,
    }

print('Metrics & inference helpers defined.')

Metrics & inference helpers defined.


## 5. Load & Freeze GRU and LSTM Base Models

In [15]:
# ── Load GRU (window=30, small) — FROZEN ─────────────────────────
gru = GRUQuantile(
    input_size=N_FEATURES, hidden_size=32,
    num_layers=1, dropout=0.10, num_quantiles=7,
).to(DEVICE)
gru.load_state_dict(torch.load(GRU_CKPT, map_location=DEVICE))
for p in gru.parameters():
    p.requires_grad = False
gru.eval()
print(f'GRU  loaded & frozen  ({sum(p.numel() for p in gru.parameters()):,} params)')

# ── Load LSTM (window=30, small) — FROZEN ────────────────────────
lstm = LSTMQuantile(
    input_size=N_FEATURES, hidden_size=32,
    num_layers=1, dropout=0.10, num_quantiles=7,
).to(DEVICE)
lstm.load_state_dict(torch.load(LSTM_CKPT, map_location=DEVICE))
for p in lstm.parameters():
    p.requires_grad = False
lstm.eval()
print(f'LSTM loaded & frozen  ({sum(p.numel() for p in lstm.parameters()):,} params)')

# ── Generate predictions ───────────────────────────────────────
print('\nGRU val predictions ...')
gru_val  = get_all_quantile_preds(gru, val30)
print('GRU test predictions ...')
gru_test = get_all_quantile_preds(gru, test30)

print('\nLSTM val predictions ...')
lstm_val  = get_all_quantile_preds(lstm, val30)
print('LSTM test predictions ...')
lstm_test = get_all_quantile_preds(lstm, test30)

print(f'\nGRU  val={gru_val["trues"].shape[0]:,}  test={gru_test["trues"].shape[0]:,}')
print(f'LSTM val={lstm_val["trues"].shape[0]:,}  test={lstm_test["trues"].shape[0]:,}')

GRU  loaded & frozen  (19,911 params)
LSTM loaded & frozen  (26,311 params)

GRU val predictions ...
GRU test predictions ...

LSTM val predictions ...
LSTM test predictions ...

GRU  val=6,027  test=9,095
LSTM val=6,027  test=9,095


## 6. TFT Data Pipeline & Frozen Inference

In [16]:
# ── TFT-specific data pipeline (mirrors tft_stock_prediction.py) ───────
RAW_FEATURE_DROP = {
    'Open', 'High', 'Low', 'Close', 'Adj Close',
    'Volume', 'symbol_base', DATE_COL,
}
TFT_KNOWN_CATS = ['dow', 'dom', 'month', 'is_month_start', 'is_month_end']


def prepare_tft_df(path):
    """Load data with TFT-specific preprocessing (categoricals as str)."""
    df = pd.read_csv(path)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')
    df = df.dropna(subset=[DATE_COL])
    df = df.sort_values([SYMBOL_COL, DATE_COL]).reset_index(drop=True)
    df = df.drop_duplicates([SYMBOL_COL, DATE_COL])
    df[TIME_IDX_COL] = df[TIME_IDX_COL].astype(np.int64)
    # TFT needs categoricals as strings for embedding layers
    df['dow']            = df[DATE_COL].dt.weekday.astype(np.int16).astype(str)
    df['dom']            = df[DATE_COL].dt.day.astype(np.int16).astype(str)
    df['month']          = df[DATE_COL].dt.month.astype(np.int16).astype(str)
    df['is_month_start'] = df[DATE_COL].dt.is_month_start.astype(np.int8).astype(str)
    df['is_month_end']   = df[DATE_COL].dt.is_month_end.astype(np.int8).astype(str)
    return df


def build_tft_datasets(df, enc_len=15, pred_len=1):
    """Build TFT TimeSeriesDataSets for val and test splits."""
    train_mask = df[DATE_COL] <= pd.Timestamp(TRAIN_END)
    min_train  = enc_len + pred_len + 1
    train_counts = df.loc[train_mask].groupby(SYMBOL_COL).size()
    eligible = [s for s in sorted(df[SYMBOL_COL].unique())
                if train_counts.get(s, 0) >= min_train]
    work = df[df[SYMBOL_COL].isin(eligible)].copy()

    known_cats  = [c for c in TFT_KNOWN_CATS if c in work.columns]
    known_reals = [TIME_IDX_COL]
    numeric = [c for c in work.columns
               if pd.api.types.is_numeric_dtype(work[c])]
    excl = set(known_reals + [TARGET_COL, TIME_IDX_COL])
    unk_covs = [c for c in numeric
                if c not in excl and c not in RAW_FEATURE_DROP]
    unk_reals = [TARGET_COL] + unk_covs
    seen = set()
    unk_reals = [c for c in unk_reals if not (c in seen or seen.add(c))]

    train_df = work.loc[work[DATE_COL] <= pd.Timestamp(TRAIN_END)].copy()
    val_df   = work.loc[work[DATE_COL] <= pd.Timestamp(VAL_END)].copy()
    test_df  = work.copy()

    drop = [c for c in RAW_FEATURE_DROP
            if c in train_df.columns and c != DATE_COL]
    train_m = train_df.drop(columns=drop, errors='ignore')
    val_m   = val_df.drop(columns=drop, errors='ignore')
    test_m  = test_df.drop(columns=drop, errors='ignore')

    training = TimeSeriesDataSet(
        train_m, time_idx=TIME_IDX_COL, target=TARGET_COL,
        group_ids=[SYMBOL_COL],
        min_encoder_length=enc_len, max_encoder_length=enc_len,
        min_prediction_length=pred_len, max_prediction_length=pred_len,
        static_categoricals=[SYMBOL_COL],
        time_varying_known_categoricals=known_cats,
        time_varying_known_reals=known_reals,
        time_varying_unknown_reals=unk_reals,
        target_normalizer=GroupNormalizer(
            groups=[SYMBOL_COL], method='standard'),
        add_relative_time_idx=False, add_target_scales=True,
        add_encoder_length=True, allow_missing_timesteps=True,
    )
    vs_idx = int(work.loc[
        work[DATE_COL] >= pd.Timestamp(VAL_START), TIME_IDX_COL].min())
    ts_idx = int(work.loc[
        work[DATE_COL] >= pd.Timestamp(TEST_START), TIME_IDX_COL].min())

    val_ds  = TimeSeriesDataSet.from_dataset(
        training, val_m, min_prediction_idx=vs_idx,
        stop_randomization=True)
    test_ds = TimeSeriesDataSet.from_dataset(
        training, test_m, min_prediction_idx=ts_idx,
        stop_randomization=True)
    return training, val_ds, test_ds, work


print('Preparing TFT data ...')
tft_df = prepare_tft_df(DATA_PATH)
tft_train_ds, tft_val_ds, tft_test_ds, tft_work = build_tft_datasets(
    tft_df, enc_len=TFT_WINDOW)
print(f'TFT val samples : {len(tft_val_ds):,}')
print(f'TFT test samples: {len(tft_test_ds):,}')

Preparing TFT data ...
TFT val samples : 6,027
TFT test samples: 9,061


In [17]:
# ── Load TFT from checkpoint ───────────
import torch.serialization
import pandas.core.internals.managers
import pandas._libs.internals
import numpy # Import numpy to ensure numpy._core is accessible
torch.serialization.add_safe_globals([GroupNormalizer, pd.DataFrame, pandas.core.internals.managers.BlockManager, pandas._libs.internals._unpickle_block, numpy.ndarray, numpy.dtype, numpy.float64])

# Manually load the checkpoint to ensure CPU mapping and explicit instantiation
checkpoint = torch.load(TFT_CKPT, map_location='cpu', weights_only=False)

# Extract hyperparameters (hparams) and state_dict
hparams_from_ckpt = checkpoint['hyper_parameters']

# Instantiate the model with the loaded hyperparameters.
# TemporalFusionTransformer expects model parameters directly, unpacked from hparams.
tft_model = TemporalFusionTransformer(**hparams_from_ckpt)

# The model will implicitly be on CPU since DEVICE is 'cpu' and CUDA is not available.
# Explicitly moving it via .to(DEVICE) can trigger unnecessary CUDA checks.
# tft_model = tft_model.to(DEVICE) # Removed this problematic line

# Load the state_dict. PyTorch Lightning often prefixes keys with 'model.'.
state_dict = checkpoint['state_dict']
new_state_dict = {}
for k, v in state_dict.items():
    # Remove 'model.' prefix added by PyTorch Lightning if present
    if k.startswith('model.'):
        new_state_dict[k[6:]] = v
    else:
        new_state_dict[k] = v

tft_model.load_state_dict(new_state_dict)

for p in tft_model.parameters():
    p.requires_grad = False
tft_model.eval()
print(f'TFT  loaded & frozen  ' \
      f'({sum(p.numel() for p in tft_model.parameters()):,} params)')


@torch.no_grad()
def get_tft_predictions(model, dataset, work_df, batch_size=64):
    """Get ALL 7 quantile predictions from TFT + aligned metadata."""
    loader = dataset.to_dataloader(
        train=False, batch_size=batch_size, num_workers=NUM_WORKERS)

    pred_output = model.predict(
        loader, mode='prediction', return_index=True,
        trainer_kwargs={'logger': False, 'enable_checkpointing': False})

    # Extract predictions tensor and index DataFrame
    if hasattr(pred_output, 'output'):
        preds = pred_output.output
        idx_df = pred_output.index.copy()
    elif isinstance(pred_output, tuple):
        preds, idx_df = None, None
        for item in pred_output:
            if isinstance(item, pd.DataFrame):
                idx_df = item.copy()
            elif torch.is_tensor(item) or isinstance(item, np.ndarray):
                preds = item
            elif hasattr(item, 'output'):
                preds = item.output
                idx_df = getattr(item, 'index', idx_df)
                if idx_df is not None:
                    idx_df = idx_df.copy()
    else:
        preds = pred_output
        idx_df = None

    if torch.is_tensor(preds):
        preds = preds.cpu().numpy()

    # Shape: (N, pred_len, num_quantiles) -> (N, 7) for pred_len=1
    if preds.ndim == 3:
        preds_q = preds[:, 0, :]     # all 7 quantiles for horizon 0
    elif preds.ndim == 2:
        preds_q = preds
    else:
        preds_q = preds[:, None]

    # Identify symbol and time columns in the index
    sym_col = SYMBOL_COL
    for c in [SYMBOL_COL, '__group_id__Symbol']:
        if c in idx_df.columns:
            sym_col = c
            break
    time_col = TIME_IDX_COL
    for c in [TIME_IDX_COL, 'decoder_time_idx', '__time_idx__']:
        if c in idx_df.columns:
            time_col = c
            break

    n = len(preds_q)
    symbols   = idx_df[sym_col].astype(str).values[:n]
    time_idxs = idx_df[time_col].astype(np.int64).values[:n]

    # Look up ground truth and prices from original data
    lk_tgt = work_df.set_index([SYMBOL_COL, TIME_IDX_COL])[TARGET_COL].to_dict()
    lk_px  = work_df.set_index([SYMBOL_COL, TIME_IDX_COL])[PRICE_COL].to_dict()
    lk_dt  = (work_df.drop_duplicates(TIME_IDX_COL)
              .set_index(TIME_IDX_COL)[DATE_COL].to_dict())

    trues   = np.array([lk_tgt.get((s, int(t)), np.nan)
                        for s, t in zip(symbols, time_idxs)], dtype=np.float32)
    base_px = np.array([lk_px.get((s, int(t) - 1), np.nan)
                        for s, t in zip(symbols, time_idxs)], dtype=np.float32)
    true_px = np.array([lk_px.get((s, int(t)), np.nan)
                        for s, t in zip(symbols, time_idxs)], dtype=np.float32)
    dates   = np.array([lk_dt.get(int(t), pd.NaT)
                        for t in time_idxs])

    return {
        'preds_q':  preds_q.astype(np.float32),
        'trues':    trues,
        'base_px':  base_px,
        'true_px':  true_px,
        'symbols':  symbols,
        'dates':    dates,
    }


print('\nTFT val predictions ...')
tft_val = get_tft_predictions(tft_model, tft_val_ds, tft_work)
print(f'  Val samples: {tft_val["trues"].shape[0]:,}')

print('TFT test predictions ...')
tft_test = get_tft_predictions(tft_model, tft_test_ds, tft_work)
print(f'  Test samples: {tft_test["trues"].shape[0]:,}')

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


TFT  loaded & frozen  (155,715 params)

TFT val predictions ...


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


  Val samples: 6,027
TFT test predictions ...
  Test samples: 9,061


## 7. Align Predictions Across Models

In [19]:
def build_pred_df(pred_dict, model_name):
    """Convert prediction dict into a DataFrame keyed by (symbol, date)."""
    n = len(pred_dict['trues'])
    df = pd.DataFrame({
        'symbol':  pred_dict['symbols'][:n],
        'date':    pd.to_datetime(pred_dict['dates'][:n]),
        'true':    pred_dict['trues'][:n],
        'base_px': pred_dict['base_px'][:n],
        'true_px': pred_dict['true_px'][:n],
    })
    preds_q = pred_dict['preds_q'][:n]

    # Handle both (N, 7) and (N, 1) shapes
    if preds_q.ndim == 1:
        preds_q = preds_q[:, None]  # (N,) -> (N, 1)

    n_q = preds_q.shape[1]
    for qi in range(n_q):
        df[f'{model_name}_q{qi}'] = preds_q[:, qi]

    if n_q == 1:
        # Only 1 quantile (median) — replicate to all 7 slots
        for qi in range(NUM_QUANTILES):
            df[f'{model_name}_q{qi}'] = preds_q[:, 0]
        df[f'{model_name}_median'] = preds_q[:, 0]
    else:
        median_idx = min(MEDIAN_IDX, n_q - 1)
        df[f'{model_name}_median'] = preds_q[:, median_idx]

    return df


def align_predictions(lstm_p, gru_p, tft_p):
    """Inner-join predictions by (symbol, date) across all 3 models."""
    df_lstm = build_pred_df(lstm_p, 'lstm')
    df_gru  = build_pred_df(gru_p,  'gru')
    df_tft  = build_pred_df(tft_p,  'tft')

    merged = df_lstm.merge(
        df_gru.drop(columns=['true', 'base_px', 'true_px']),
        on=['symbol', 'date'], how='inner'
    ).merge(
        df_tft.drop(columns=['true', 'base_px', 'true_px']),
        on=['symbol', 'date'], how='inner'
    )
    merged = merged.dropna().reset_index(drop=True)
    print(f'  Aligned: {len(merged):,} samples | '
          f'{merged["symbol"].nunique()} symbols')
    return merged


print('Aligning val predictions ...')
val_aligned  = align_predictions(lstm_val, gru_val, tft_val)
print('Aligning test predictions ...')
test_aligned = align_predictions(lstm_test, gru_test, tft_test)


Aligning val predictions ...
  Aligned: 6,027 samples | 49 symbols
Aligning test predictions ...
  Aligned: 9,061 samples | 49 symbols


In [20]:
def extract_stack_features(aligned_df):
    """Extract 21 features (7 quantiles x 3 models) from aligned df."""
    feat_cols = []
    for mdl in ['lstm', 'gru', 'tft']:
        for qi in range(NUM_QUANTILES):
            feat_cols.append(f'{mdl}_q{qi}')
    X       = aligned_df[feat_cols].values.astype(np.float32)
    y       = aligned_df['true'].values.astype(np.float32)
    base_px = aligned_df['base_px'].values.astype(np.float32)
    true_px = aligned_df['true_px'].values.astype(np.float32)
    return X, y, base_px, true_px

X_val,  y_val,  bp_val,  tp_val  = extract_stack_features(val_aligned)
X_test, y_test, bp_test, tp_test = extract_stack_features(test_aligned)
print(f'Val  stacked features: {X_val.shape}')
print(f'Test stacked features: {X_test.shape}')

Val  stacked features: (6027, 21)
Test stacked features: (9061, 21)


## 8. MLP Meta-Learner (Stacking Head)

In [21]:
class StackingMLP(nn.Module):
    """MLP meta-learner for stacking ensemble.

    Input : 21 features (7 quantiles x 3 models)
    Output: 1  (next-day return prediction)
    """
    def __init__(self, input_dim=21, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(32, 16),
            nn.GELU(),

            nn.Linear(16, 1),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x).squeeze(-1)


meta_model = StackingMLP(input_dim=X_val.shape[1]).to(DEVICE)
total_p = sum(p.numel() for p in meta_model.parameters())
train_p = sum(p.numel() for p in meta_model.parameters()
              if p.requires_grad)
print(f'MLP Meta-Learner: {total_p:,} total | {train_p:,} trainable')
print(meta_model)

MLP Meta-Learner: 4,225 total | 4,225 trainable
StackingMLP(
  (net): Sequential(
    (0): Linear(in_features=21, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): GELU(approximate='none')
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=32, out_features=16, bias=True)
    (9): GELU(approximate='none')
    (10): Linear(in_features=16, out_features=1, bias=True)
  )
)


## 9. Train Meta-Learner on Val Predictions

In [22]:
# Split val into meta-train (70%) and meta-val (30%)
n_val = len(X_val)
n_mt  = int(0.7 * n_val)
perm  = np.random.permutation(n_val)
mt_idx, mv_idx = perm[:n_mt], perm[n_mt:]

X_mt = torch.tensor(X_val[mt_idx], device=DEVICE)
y_mt = torch.tensor(y_val[mt_idx], device=DEVICE)
X_mv = torch.tensor(X_val[mv_idx], device=DEVICE)
y_mv = torch.tensor(y_val[mv_idx], device=DEVICE)

mt_ds = TensorDataset(X_mt, y_mt)
mt_dl = DataLoader(mt_ds, batch_size=META_BS,
                   shuffle=True, drop_last=True)

optimizer = optim.Adam(meta_model.parameters(),
                       lr=META_LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 'min', factor=0.5, patience=5, min_lr=1e-6)
criterion = nn.MSELoss()

best_mv_loss = float('inf')
patience_cnt = 0
best_state   = None
history = {'train_loss': [], 'val_loss': []}

pbar = tqdm(range(1, META_EPOCHS + 1), desc='Meta-Train')
for epoch in pbar:
    meta_model.train()
    t_sum, t_n = 0.0, 0
    for xb, yb in mt_dl:
        optimizer.zero_grad()
        loss = criterion(meta_model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(meta_model.parameters(), 1.0)
        optimizer.step()
        t_sum += loss.item(); t_n += 1
    t_loss = t_sum / max(t_n, 1)

    meta_model.eval()
    with torch.no_grad():
        v_loss = criterion(meta_model(X_mv), y_mv).item()

    scheduler.step(v_loss)
    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    pbar.set_postfix(train=f'{t_loss:.6f}',
                     val=f'{v_loss:.6f}',
                     best=f'{best_mv_loss:.6f}')

    if v_loss < best_mv_loss - 1e-7:
        best_mv_loss = v_loss
        patience_cnt = 0
        best_state = {k: v.clone()
                      for k, v in meta_model.state_dict().items()}
        torch.save(meta_model.state_dict(),
                   OUTPUT_DIR / 'checkpoints' / 'meta_mlp_best.pt')
    else:
        patience_cnt += 1
        if patience_cnt >= META_PATIENCE:
            print(f'  Early stop at epoch {epoch}')
            break

if best_state is not None:
    meta_model.load_state_dict(best_state)
meta_model.eval()
print(f'Training complete. Best meta-val MSE: {best_mv_loss:.6f}')

Meta-Train:   0%|          | 0/100 [00:00<?, ?it/s]

  Early stop at epoch 22
Training complete. Best meta-val MSE: 0.000351


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# ENSEMBLE META-LEARNER COMPARISON — Multiple Stacking Strategies
# ══════════════════════════════════════════════════════════════════════

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler as SKScaler

# ── 1. Optimized Inverse-Variance Weighting (no training needed) ──────
# Weight each model by 1 / val_MSE — simple but surprisingly effective.

val_mse_lstm = np.mean((val_aligned['lstm_median'].values - y_val) ** 2)
val_mse_gru  = np.mean((val_aligned['gru_median'].values  - y_val) ** 2)
val_mse_tft  = np.mean((val_aligned['tft_median'].values  - y_val) ** 2)

inv_var = np.array([1/val_mse_lstm, 1/val_mse_gru, 1/val_mse_tft])
inv_var_weights = inv_var / inv_var.sum()
print(f'Inverse-Variance Weights: LSTM={inv_var_weights[0]:.4f}, '
      f'GRU={inv_var_weights[1]:.4f}, TFT={inv_var_weights[2]:.4f}')

ivw_preds = (test_aligned['lstm_median'].values * inv_var_weights[0] +
             test_aligned['gru_median'].values  * inv_var_weights[1] +
             test_aligned['tft_median'].values  * inv_var_weights[2])


# ── 2. Ridge Regression (L2-regularized linear) ──────────────────────
# Often the BEST meta-learner — avoids overfitting on noisy financial data.

ridge = Pipeline([
    ('scaler', SKScaler()),
    ('ridge', Ridge(alpha=1.0))
])
ridge.fit(X_val, y_val)
ridge_preds = ridge.predict(X_test)


# ── 3. ElasticNet (L1+L2) — automatic feature selection ─────────────
# Sparsity from L1 can zero out useless quantiles.

enet = Pipeline([
    ('scaler', SKScaler()),
    ('enet', ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000))
])
enet.fit(X_val, y_val)
enet_preds = enet.predict(X_test)


# ── 4. Gradient Boosting (XGBoost-like) ──────────────────────────────
# Best tree-based method for tabular data; handles non-linearities.

gbr = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=3,            # shallow trees to avoid overfitting
    learning_rate=0.05,
    subsample=0.8,
    min_samples_leaf=20,
    random_state=SEED,
)
gbr.fit(X_val, y_val)
gbr_preds = gbr.predict(X_test)


# ── 5. Random Forest ─────────────────────────────────────────────────
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=20,
    random_state=SEED,
    n_jobs=-1,
)
rf.fit(X_val, y_val)
rf_preds = rf.predict(X_test)


# ── 6. SVR (Support Vector Regression) ───────────────────────────────
svr = Pipeline([
    ('scaler', SKScaler()),
    ('svr', SVR(kernel='rbf', C=1.0, epsilon=0.001))
])
svr.fit(X_val, y_val)
svr_preds = svr.predict(X_test)


# ── 7. Blended Ensemble (average of top meta-learners) ───────────────
# Often the most robust: average multiple meta-learners to reduce variance.
blend_preds = (ridge_preds + gbr_preds + ensemble_preds) / 3.0


# ══════════════════════════════════════════════════════════════════════
# EVALUATE ALL METHODS
# ══════════════════════════════════════════════════════════════════════

all_results = {}
for name, pred in [
    ('LSTM (w30)',          test_aligned['lstm_median'].values),
    ('GRU  (w30)',          test_aligned['gru_median'].values),
    ('TFT  (w15)',          test_aligned['tft_median'].values),
    ('Simple Average',      simple_avg),
    ('Inv-Variance Wt',     ivw_preds),
    ('Ridge (L2)',          ridge_preds),
    ('ElasticNet (L1+L2)',  enet_preds),
    ('Gradient Boosting',   gbr_preds),
    ('Random Forest',       rf_preds),
    ('SVR (RBF)',           svr_preds),
    ('Stacking MLP',        ensemble_preds),
    ('Blended Top-3',       blend_preds),
]:
    m = compute_metrics(y_test, pred, bp_test, tp_test)
    all_results[name] = m

# Summary table
comp_df = pd.DataFrame(all_results).T
comp_df = comp_df[[c for c in REPORT_COLS if c in comp_df.columns]].round(4)
display(HTML(comp_df.to_html()))
comp_df.to_csv(OUTPUT_DIR / 'results' / 'full_ensemble_comparison.csv')
print(f'\nSaved to {OUTPUT_DIR / "results" / "full_ensemble_comparison.csv"}')


In [23]:
# ── Training curve ───────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(history['train_loss'], label='Meta-Train', alpha=0.8)
ax.plot(history['val_loss'],   label='Meta-Val',   alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Meta-Learner Training Curve')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'plots' / 'meta_training_loss.png', dpi=150)
plt.show()

## 10. Evaluate Ensemble on Test Set

In [24]:
# ── Generate ensemble predictions on test ─────────────────────────
meta_model.eval()
X_test_t = torch.tensor(X_test, device=DEVICE)
with torch.no_grad():
    ensemble_preds = meta_model(X_test_t).cpu().numpy()

# Individual model medians on aligned test
lstm_med = test_aligned['lstm_median'].values
gru_med  = test_aligned['gru_median'].values
tft_med  = test_aligned['tft_median'].values
simple_avg = (lstm_med + gru_med + tft_med) / 3.0

# Compute metrics for every method
results = {}
for name, pred in [
    ('LSTM (w30)',      lstm_med),
    ('GRU  (w30)',      gru_med),
    ('TFT  (w15)',      tft_med),
    ('Simple Average',  simple_avg),
    ('Stacking MLP',    ensemble_preds),
]:
    m = compute_metrics(y_test, pred, bp_test, tp_test)
    results[name] = m
    print(f'\n{name}:')
    for k in REPORT_COLS:
        if k in m:
            print(f'  {k:20s}: {m[k]:.6f}')

# Summary table
summary_df = pd.DataFrame(results).T
summary_df = summary_df[[c for c in REPORT_COLS if c in summary_df.columns]]
summary_df = summary_df.round(4)
display(HTML(summary_df.to_html()))

summary_df.to_csv(OUTPUT_DIR / 'results' / 'ensemble_comparison.csv')
print(f'\nSaved to {OUTPUT_DIR / "results" / "ensemble_comparison.csv"}')


LSTM (w30):
  MAPE_price          : 16.599977
  MSE_pct             : 2.201770
  RMSE_pct            : 1.483836
  MAE_pct             : 1.047066
  F1_macro            : 0.451750
  Accuracy            : 0.491116
  Precision_macro     : 0.501039
  Recall_macro        : 0.500703
  Directional_Acc     : 0.491116
  n_samples           : 9061.000000

GRU  (w30):
  MAPE_price          : 16.583057
  MSE_pct             : 2.203357
  RMSE_pct            : 1.484371
  MAE_pct             : 1.046364
  F1_macro            : 0.428770
  Accuracy            : 0.499172
  Precision_macro     : 0.478164
  Recall_macro        : 0.487908
  Directional_Acc     : 0.499172
  n_samples           : 9061.000000

TFT  (w15):
  MAPE_price          : 16.598883
  MSE_pct             : 2.197317
  RMSE_pct            : 1.482335
  MAE_pct             : 1.044997
  F1_macro            : 0.502731
  Accuracy            : 0.504139
  Precision_macro     : 0.506626
  Recall_macro        : 0.506503
  Directional_Acc     : 0.50

,MAPE_price,MSE_pct,RMSE_pct,MAE_pct,F1_macro,Accuracy,Precision_macro,Recall_macro,Directional_Acc,n_samples
LSTM (w30),16.6000,2.2018,1.4838,1.0471,0.4517,0.4911,0.5010,0.5007,0.4911,9061.0
GRU (w30),16.5831,2.2034,1.4844,1.0464,0.4288,0.4992,0.4782,0.4879,0.4992,9061.0
TFT (w15),16.5989,2.1973,1.4823,1.0450,0.5027,0.5041,0.5066,0.5065,0.5041,9061.0
Simple Average,16.5936,2.1984,1.4827,1.0452,0.4889,0.4918,0.4897,0.4898,0.4918,9061.0
Stacking MLP,16.5819,2.2911,1.5136,1.0638,0.5059,0.5087,0.5121,0.5118,0.5087,9061.0



Saved to ensemble_outputs/results/ensemble_comparison.csv


## 11. Comparison Plots

In [25]:
# ── Bar chart comparison ─────────────────────────────────────────
metrics_to_plot = ['MAPE_price', 'MAE_pct', 'F1_macro', 'Directional_Acc']
model_names = list(results.keys())
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, metric in enumerate(metrics_to_plot):
    ax = axes[i // 2, i % 2]
    vals = [results[n].get(metric, 0) for n in model_names]
    bars = ax.bar(range(len(model_names)), vals,
                  color=colors[:len(model_names)], alpha=0.85)
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels(model_names, rotation=30, ha='right', fontsize=7)
    ax.set_title(metric, fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

    # Highlight best
    if metric in ['MAPE_price', 'MAE_pct', 'MSE_pct', 'RMSE_pct']:
        best_idx = int(np.argmin(vals))
    else:
        best_idx = int(np.argmax(vals))
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(2.5)

    for j, v in enumerate(vals):
        ax.text(j, v, f'{v:.4f}', ha='center', va='bottom', fontsize=6)

plt.suptitle('Ensemble vs Individual Models — Test Set',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'plots' / 'ensemble_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# VISUAL COMPARISON
# ══════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
metrics_to_plot = ['MAPE_price', 'MAE_pct', 'F1_macro', 'Directional_Acc']
model_names = list(all_results.keys())

# Color: base models blue, meta-learners gradient purple→green
n = len(model_names)
cmap = plt.cm.viridis(np.linspace(0.1, 0.9, n))

for i, metric in enumerate(metrics_to_plot):
    ax = axes[i // 2, i % 2]
    vals = [all_results[nm].get(metric, 0) for nm in model_names]
    bars = ax.barh(range(n), vals, color=cmap, alpha=0.85)
    ax.set_yticks(range(n))
    ax.set_yticklabels(model_names, fontsize=7)
    ax.set_title(metric, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    ax.invert_yaxis()

    # Highlight best
    if metric in ['MAPE_price', 'MAE_pct']:
        best_idx = int(np.argmin(vals))
    else:
        best_idx = int(np.argmax(vals))
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(2.5)

    for j, v in enumerate(vals):
        ax.text(v, j, f' {v:.4f}', va='center', fontsize=6)

plt.suptitle('All Ensemble Methods — Test Set Comparison',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'plots' / 'full_ensemble_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()


In [26]:
# ── Actual vs Predicted scatter ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, pred) in zip(axes, [
    ('LSTM (w30)', lstm_med),
    ('GRU  (w30)', gru_med),
    ('Stacking MLP', ensemble_preds),
]):
    ax.scatter(y_test, pred, alpha=0.1, s=5, c='steelblue')
    lo = min(y_test.min(), pred.min())
    hi = max(y_test.max(), pred.max())
    ax.plot([lo, hi], [lo, hi], 'r--', alpha=0.5, linewidth=1)
    ax.set_xlabel('Actual Return')
    ax.set_ylabel('Predicted Return')
    ax.set_title(name)
    ax.grid(True, alpha=0.3)

plt.suptitle('Actual vs Predicted — Test Set',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'plots' / 'scatter_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [27]:
# ── Direction accuracy per symbol ─────────────────────────────────
test_aligned['ens_pred'] = ensemble_preds
test_aligned['ens_dir']  = (ensemble_preds > 0).astype(int)
test_aligned['true_dir'] = (y_test > 0).astype(int)
test_aligned['correct']  = (
    test_aligned['ens_dir'] == test_aligned['true_dir']).astype(int)

per_sym = (test_aligned.groupby('symbol')['correct']
           .mean().sort_values(ascending=False))

print('Top 10 symbols by ensemble directional accuracy:')
print(per_sym.head(10).to_string())
print(f'\nBottom 10:')
print(per_sym.tail(10).to_string())
print(f'\nPer-symbol mean DA: {per_sym.mean():.4f}')

per_sym.to_csv(OUTPUT_DIR / 'results' / 'per_symbol_direction_acc.csv')

Top 10 symbols by ensemble directional accuracy:
symbol
ETERNAL.NS       0.584699
JIOFIN.NS        0.584699
TCS.NS           0.578378
INFY.NS          0.551351
SUNPHARMA.NS     0.551351
TATACONSUM.NS    0.551351
TECHM.NS         0.551351
TATASTEEL.NS     0.545946
HCLTECH.NS       0.540541
HDFCBANK.NS      0.540541

Bottom 10:
symbol
RELIANCE.NS      0.475676
M&M.NS           0.470270
ULTRACEMCO.NS    0.470270
GRASIM.NS        0.464865
ADANIPORTS.NS    0.454054
BAJAJFINSV.NS    0.443243
ADANIENT.NS      0.437838
LT.NS            0.437838
ICICIBANK.NS     0.427027
HINDUNILVR.NS    0.421622

Per-symbol mean DA: 0.5087


In [28]:
# ── Per-symbol DA bar chart (top/bottom 10) ───────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

top10 = per_sym.head(10)
ax1.barh(range(len(top10)), top10.values, color='#2ecc71', alpha=0.8)
ax1.set_yticks(range(len(top10)))
ax1.set_yticklabels(top10.index, fontsize=8)
ax1.set_xlabel('Directional Accuracy')
ax1.set_title('Top 10 Symbols', fontweight='bold')
ax1.axvline(0.5, color='red', linestyle='--', alpha=0.5)
ax1.grid(True, alpha=0.3, axis='x')
ax1.invert_yaxis()

bot10 = per_sym.tail(10)
ax2.barh(range(len(bot10)), bot10.values, color='#e74c3c', alpha=0.8)
ax2.set_yticks(range(len(bot10)))
ax2.set_yticklabels(bot10.index, fontsize=8)
ax2.set_xlabel('Directional Accuracy')
ax2.set_title('Bottom 10 Symbols', fontweight='bold')
ax2.axvline(0.5, color='red', linestyle='--', alpha=0.5)
ax2.grid(True, alpha=0.3, axis='x')
ax2.invert_yaxis()

plt.suptitle('Ensemble Directional Accuracy by Symbol',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'plots' / 'per_symbol_da.png',
            dpi=150, bbox_inches='tight')
plt.show()

## 12. Final Summary

In [29]:
print('=' * 60)
print(' ENSEMBLE STACKING MODEL — FINAL RESULTS')
print('=' * 60)
print()
print('Base Models (all FROZEN):')
print(f'  LSTM  window=30, small (hidden=32)  ← {LSTM_CKPT}')
print(f'  GRU   window=30, small (hidden=32)  ← {GRU_CKPT}')
print(f'  TFT   window=15, hidden=32, attn=4  ← {TFT_CKPT}')
print()
print(f'Meta-Learner: StackingMLP'
      f' ({X_val.shape[1]} → 64 → 32 → 16 → 1)')
print(f'  Trained on val predictions: {X_val.shape[0]:,} aligned samples')
print(f'  Tested on: {X_test.shape[0]:,} aligned samples')
print()
print(f'{"Model":20s} | {"MAPE%":>8s} | {"MAE_pct":>8s} | '
      f'{"DA":>6s} | {"F1_macro":>8s}')
print('-' * 60)
for name in results:
    m = results[name]
    print(f'{name:20s} | '
          f'{m.get("MAPE_price",0):8.4f} | '
          f'{m.get("MAE_pct",0):8.4f} | '
          f'{m.get("Directional_Acc",0):6.4f} | '
          f'{m.get("F1_macro",0):8.4f}')
print('=' * 60)
print(f'All outputs saved to: {OUTPUT_DIR}')

 ENSEMBLE STACKING MODEL — FINAL RESULTS

Base Models (all FROZEN):
  LSTM  window=30, small (hidden=32)  ← /content/drive/MyDrive/inlp/lstm_outputs/lstm_outputs-2/checkpoints/w30_small.pt
  GRU   window=30, small (hidden=32)  ← /content/drive/MyDrive/inlp/GRU_outputs/gru_outputs/checkpoints/w30_small.pt
  TFT   window=15, hidden=32, attn=4  ← /content/drive/MyDrive/inlp/tft/window_15/checkpoints/best-epoch003-valloss0.007141.ckpt

Meta-Learner: StackingMLP (21 → 64 → 32 → 16 → 1)
  Trained on val predictions: 6,027 aligned samples
  Tested on: 9,061 aligned samples

Model                |    MAPE% |  MAE_pct |     DA | F1_macro
------------------------------------------------------------
LSTM (w30)           |  16.6000 |   1.0471 | 0.4911 |   0.4517
GRU  (w30)           |  16.5831 |   1.0464 | 0.4992 |   0.4288
TFT  (w15)           |  16.5989 |   1.0450 | 0.5041 |   0.5027
Simple Average       |  16.5936 |   1.0452 | 0.4918 |   0.4889
Stacking MLP         |  16.5819 |   1.0638 | 0.508

In [30]:
import shutil
from google.colab import files

# Zip the output directory
output_zip_name = 'ensemble_outputs.zip'
shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)

print(f'Zipped output to {output_zip_name}')

# Offer the zipped file for download
files.download(output_zip_name)

Zipped output to ensemble_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>